# Phase 3 — Initial Gold Build

Run this notebook **once** after confirming that `silver_weather` contains the intended Phase 2 production data.

It loads the reusable functions from `10_Silver_to_Gold` and runs a full Gold baseline. It does not call the Open-Meteo API or modify Bronze/Silver. After it finishes, run `11_Validate_Gold.ipynb`.


In [0]:
%run ./10_Silver_to_Gold


# Phase 3 — Silver to Gold Lakehouse

## Purpose
Materialize the Gold star schema and analytics tables using the existing Phase 2 Delta tables.

## Tables created
- `dim_city` — one row per city; deterministic `city_key`.
- `dim_date` — one row per calendar date; `date_key` uses `yyyyMMdd`.
- `fact_weather` — one row per city per hourly timestamp; logical key is `city_key + timestamp`.
- `gold_daily_weather`, `gold_monthly_climate`, `gold_annual_climate`, `gold_extreme_weather`.

## Integration
1. Place this notebook in the same Databricks workspace folder as the Full Load and Incremental Load notebooks.
2. Near the end of each load notebook, run `%run ./10_Silver_to_Gold`.
3. After the load's city-year loop completes, call `run_gold_pipeline(load_type="FULL")` in the full-load notebook and `run_gold_pipeline(load_type="INCREMENTAL")` in the incremental notebook.
4. Do not call Gold inside every city-year loop; refresh once after the loop to avoid repeatedly aggregating all Silver data.

## Design notes
- Uses the table names and weather columns from the uploaded Phase 2 notebooks.
- `observation_year` is a partition column on first creation of `fact_weather` to support time-range filters. If `fact_weather` already exists unpartitioned, inspect/migrate deliberately rather than dropping it.
- Fact uses Delta `MERGE`; incremental runs use the last successful Gold fact-log timestamp as a watermark and merge Silver rows whose `load_timestamp` is newer. A `FULL` run reprocesses the complete Silver baseline. Aggregate tables are rebuilt with idempotent overwrite.
- Extreme-weather thresholds are project-defined indicators, not universal official climate thresholds.
- The logging helper fills `rows_aggregated` / duration fields if present in the existing log schema. Otherwise it records the output count in `rows_inserted` and timing in the existing start/end fields.


Gold configuration loaded. Source: silver_weather


Gold logging helpers ready.


Dimension functions ready.


Fact-table MERGE function ready.


Aggregate-building functions ready.


run_gold_pipeline(load_type='FULL' or 'INCREMENTAL') is ready.


Optional OPTIMIZE skipped. Set RUN_GOLD_OPTIMIZE = True after validation.


In [0]:
run_gold_pipeline(load_type="FULL")


SILVER → GOLD PIPELINE | run_id: 6df04201-d87b-4e2e-aae2-b3043720a9b0 | load_type: FULL
Silver rows available: 3,550,176
dim_city: SUCCESS | output=9 | inserted=9 | updated=0 | seconds=21.51
dim_date: SUCCESS | output=16,436 | inserted=16,436 | updated=0 | seconds=9.99
fact_weather: SUCCESS | output=3,550,176 | inserted=3,550,176 | updated=0 | seconds=15.63
gold_daily_weather: SUCCESS | output=147,924 | inserted=147,924 | updated=0 | seconds=5.89
gold_monthly_climate: SUCCESS | output=4,860 | inserted=4,860 | updated=0 | seconds=5.90
gold_annual_climate: SUCCESS | output=405 | inserted=405 | updated=0 | seconds=4.30
gold_extreme_weather: SUCCESS | output=147,924 | inserted=147,924 | updated=0 | seconds=4.23
GOLD REFRESH COMPLETED.
Fact: fact_weather
Dimensions: dim_city dim_date
Aggregates: gold_daily_weather gold_monthly_climate gold_annual_climate gold_extreme_weather
